## 1. Что такое NLP

**Natural Language Processing** — область ИИ, которая работает с человеческим языком.

Примеры:
- анализ тональности;
- спам-фильтрация;
- перевод;
- чат-боты;
- поиск;
- суммаризация;
- генерация текста;
- классификация обращений.

Главная проблема: нейросеть не умеет принимать обычную строку напрямую. Текст нужно преобразовать в числа.

## 2. Установка библиотек

```bash
pip install nltk gensim scikit-learn matplotlib
```

In [1]:
# !pip install nltk gensim scikit-learn matplotlib

## 3. NLTK

NLTK — классическая Python-библиотека для обработки текста.

Сегодня используем её для:
- токенизации;
- stop words;
- stemming;
- lemmatization.

In [2]:
import nltk

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

print("NLTK готов")

NLTK готов


## 4. Токенизация

Токенизация — разбиение текста на токены.

```text
"Machine learning is interesting!"
↓
["Machine", "learning", "is", "interesting", "!"]
```

Токен — не обязательно целое слово. Это единица текста, которую выделяет токенизатор.

In [3]:
from nltk.tokenize import word_tokenize, sent_tokenize

text = "Machine learning is interesting. NLP works with human language!"

print(sent_tokenize(text))
print(word_tokenize(text))

['Machine learning is interesting.', 'NLP works with human language!']
['Machine', 'learning', 'is', 'interesting', '.', 'NLP', 'works', 'with', 'human', 'language', '!']


### Почему не просто `.split()`?

`split()` разбивает только по пробелам и хуже работает с пунктуацией.

In [4]:
text = "Hello, world! NLP is cool."

print("split():", text.split())
print("NLTK:", word_tokenize(text))

split(): ['Hello,', 'world!', 'NLP', 'is', 'cool.']
NLTK: ['Hello', ',', 'world', '!', 'NLP', 'is', 'cool', '.']


## 5. Stop words

Стоп-слова — очень частые служебные слова, которые в некоторых задачах несут мало полезного сигнала.

Но удалять их нужно осознанно.

Например:

```text
I do not like this game
```

Если удалить `not`, смысл изменится.

In [5]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
text = "This is a very simple example of natural language processing"

tokens = word_tokenize(text.lower())
filtered = [t for t in tokens if t.isalpha() and t not in stop_words]

print(tokens)
print(filtered)

['this', 'is', 'a', 'very', 'simple', 'example', 'of', 'natural', 'language', 'processing']
['simple', 'example', 'natural', 'language', 'processing']


## 6. Stemming

Stemming — грубое приведение слова к основе.

Он может вернуть форму, которая не является настоящим словом.

In [6]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

for word in ["playing", "played", "studies", "studying", "connected", "connection"]:
    print(f"{word:12} -> {stemmer.stem(word)}")

playing      -> play
played       -> play
studies      -> studi
studying     -> studi
connected    -> connect
connection   -> connect


## 7. Lemmatization

Лемматизация пытается получить словарную форму слова — лемму.

Она обычно лингвистически осмысленнее stemming.

In [7]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

for word in ["cars", "mice", "running", "better"]:
    print(f"{word:10} -> {lemmatizer.lemmatize(word)}")

print("running как глагол ->", lemmatizer.lemmatize("running", pos="v"))
print("better как прилагательное ->", lemmatizer.lemmatize("better", pos="a"))

cars       -> car
mice       -> mouse
running    -> running
better     -> better
running как глагол -> run
better как прилагательное -> good


## 8. Stemming vs Lemmatization

| Stemming | Lemmatization |
|---|---|
| грубо обрезает | ищет словарную форму |
| быстрее | обычно сложнее |
| может вернуть несуществующую форму | старается вернуть реальное слово |
| мало грамматики | может учитывать часть речи |

## 9. Простой preprocessing pipeline

```text
RAW TEXT
↓
lowercase
↓
tokenization
↓
remove punctuation
↓
remove stop words
↓
lemmatization
```

In [8]:
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    result = []

    for token in tokens:
        if not token.isalpha():
            continue
        if token in stop_words:
            continue

        result.append(lemmatizer.lemmatize(token))

    return result

example = "Natural language processing allows computers to work with human languages."
print(preprocess_text(example))

['natural', 'language', 'processing', 'allows', 'computer', 'work', 'human', 'language']


## 10. Корпус

**Corpus** — коллекция текстов.

```text
document = один текст
corpus = много документов
```

Корпусами могут быть:
- новости;
- книги;
- отзывы;
- Wikipedia;
- сообщения;
- документация.

На корпусах обучают NLP-модели и строят статистику языка.

In [9]:
corpus = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "natural language processing works with text",
    "computer vision works with images",
    "neural networks learn useful representations"
]

for i, document in enumerate(corpus, 1):
    print(i, document)

1 machine learning is interesting
2 deep learning uses neural networks
3 natural language processing works with text
4 computer vision works with images
5 neural networks learn useful representations


## 11. Частотный анализ корпуса

In [10]:
from collections import Counter

all_words = []

for document in corpus:
    all_words.extend(word_tokenize(document.lower()))

print(Counter(all_words).most_common(10))

[('learning', 2), ('neural', 2), ('networks', 2), ('works', 2), ('with', 2), ('machine', 1), ('is', 1), ('interesting', 1), ('deep', 1), ('uses', 1)]


## 12. Как превратить слова в числа?

Самая простая идея — One-Hot Encoding.

```text
cat = [1,0,0]
dog = [0,1,0]
car = [0,0,1]
```

Проблемы:
1. при большом словаре векторы огромные;
2. one-hot не показывает смысловую близость.

Для него `cat` не ближе к `dog`, чем к `car`.

## 13. Embeddings

Embedding — плотный числовой вектор, представляющий слово.

```text
cat -> [0.23, -0.71, 0.18, ...]
dog -> [0.19, -0.68, 0.22, ...]
```

Идея:

> слова, встречающиеся в похожих контекстах, должны получать похожие векторы.

Embedding можно воспринимать как координаты слова в многомерном смысловом пространстве.

## 14. Cosine Similarity

Для сравнения embeddings часто используют косинусное сходство:

```text
cos(A,B) = (A·B) / (||A|| ||B||)
```

Условно:
- ближе к 1 — направления похожи;
- около 0 — мало связи;
- ближе к -1 — противоположны.

In [11]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

cat = np.array([[0.8, 0.7, 0.1]])
dog = np.array([[0.75, 0.72, 0.15]])
car = np.array([[0.1, 0.2, 0.9]])

print("cat vs dog:", cosine_similarity(cat, dog)[0][0])
print("cat vs car:", cosine_similarity(cat, car)[0][0])

cat vs dog: 0.9977257272844682
cat vs car: 0.313083400758126


## 15. Word2Vec

Word2Vec обучает embeddings слов на основе контекста.

Пример:

```text
I drink hot coffee
I drink hot tea
```

`coffee` и `tea` используются в похожем окружении, поэтому модель может сделать их embeddings похожими.

## 16. CBOW и Skip-gram

### CBOW

```text
context -> central word
```

Например:

```text
I ___ Python every day
```

По контексту модель предсказывает центральное слово.

### Skip-gram

```text
central word -> context
```

По слову модель пытается предсказать его соседей.

В gensim:

```text
sg=0 -> CBOW
sg=1 -> Skip-gram
```

## 17. Обучим свой Word2Vec

Важно: учебный корпус маленький, поэтому результаты не будут такими качественными, как у моделей, обученных на миллионах текстов.

In [12]:
from gensim.models import Word2Vec

sentences = [
    ["cat", "animal", "pet", "cute"],
    ["dog", "animal", "pet", "friendly"],
    ["cat", "pet", "home", "animal"],
    ["dog", "pet", "home", "animal"],
    ["lion", "animal", "wild", "strong"],
    ["tiger", "animal", "wild", "strong"],
    ["car", "vehicle", "road", "drive"],
    ["truck", "vehicle", "road", "drive"],
    ["bus", "vehicle", "road", "transport"],
    ["python", "programming", "language", "code"],
    ["java", "programming", "language", "code"],
    ["javascript", "programming", "language", "web"],
] * 30

model = Word2Vec(
    sentences=sentences,
    vector_size=20,
    window=3,
    min_count=1,
    workers=1,
    sg=1,
    epochs=100
)

print(model.wv["cat"][:5])

[-0.36726117 -0.08625377  0.118278    0.16033353  0.23248704]


## 18. Ищем похожие слова

In [13]:
for word in ["cat", "dog", "car", "python"]:
    print(f"\nПохожие на {word}:")
    for similar_word, score in model.wv.most_similar(word, topn=5):
        print(f"{similar_word:12} {score:.3f}")


Похожие на cat:
dog          0.989
animal       0.971
friendly     0.969
cute         0.965
home         0.961

Похожие на dog:
cat          0.989
friendly     0.976
cute         0.974
home         0.972
animal       0.958

Похожие на car:
truck        0.996
transport    0.989
vehicle      0.985
bus          0.984
road         0.982

Похожие на python:
java         0.995
programming  0.982
web          0.979
language     0.970
code         0.970


## 19. Сравниваем слова напрямую

In [14]:
for a, b in [("cat","dog"), ("cat","car"), ("python","java"), ("car","truck")]:
    print(f"{a:10} <-> {b:10}: {model.wv.similarity(a,b):.3f}")

cat        <-> dog       : 0.989
cat        <-> car       : 0.363
python     <-> java      : 0.995
car        <-> truck     : 0.996


## 20. Основные параметры Word2Vec

- `vector_size` — размер embedding-вектора;
- `window` — размер контекста;
- `min_count` — минимальная частота слова;
- `sg=0` — CBOW;
- `sg=1` — Skip-gram.

## 21. GloVe

**GloVe — Global Vectors for Word Representation.**

Word2Vec в основном учится через локальный контекст.

GloVe использует статистику того, как часто слова встречаются вместе во всём корпусе.

Упрощённо:

| | animal | pet | road | vehicle |
|---|---:|---:|---:|---:|
| cat | 500 | 700 | 2 | 1 |
| dog | 520 | 690 | 4 | 2 |
| car | 3 | 1 | 800 | 600 |

`cat` и `dog` имеют похожий профиль совместной встречаемости.

И Word2Vec, и GloVe в итоге дают:

```text
word -> vector
```

## 22. Где embeddings применяются

Например, хотим классифицировать отзыв:

```text
"This game is amazing"
```

Получаем токены:

```text
["game", "amazing"]
```

Преобразуем их в vectors:

```text
game -> vector
amazing -> vector
```

И уже эти числа можно передать нейросети.

## 23. Мини-практика

1. Создайте корпус минимум из 30–50 предложений на одну тему.
2. Выполните токенизацию.
3. Уберите пунктуацию.
4. При необходимости удалите stop words.
5. Обучите Word2Vec.
6. Для трёх слов найдите `most_similar`.
7. Сравните CBOW и Skip-gram.
8. Измените `vector_size` и `window`.
9. Объясните, почему результаты на маленьком корпусе могут быть странными.